        # 03 リタイア後の資産取り崩し

        - 目的：FIRE 達成後、**資産が何年持つか**を調べる（RQ4・RQ5）
        - 計算：`retirement_model.py`

> 架空のデータによる**教育用シミュレーション**です。将来の投資収益の予測・保証や、投資・金融商品の推奨ではありません。金額は**万円**。税・社会保険料・インフレ・手数料・年金は考慮していません。

## 準備：実験環境（最初に必ず実行）

F2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                               # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f2-financial-independence"   # リポジトリの中の F2 フォルダ（ルートが F2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f2_root(p):
    return (Path(p) / "financial_model.py").exists() and (Path(p) / "fire_analysis.py").exists()


def find_f2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F2 のルート
        if is_f2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f2_upload")
    for hit in glob.glob("/content/f2_upload/**/financial_model.py", recursive=True):
        if is_f2_root(Path(hit).parent):
            return Path(hit).parent
    return None


F2_ROOT = find_f2_root()
if F2_ROOT is None:
    raise RuntimeError("F2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F2_ROOT)
if str(F2_ROOT) not in sys.path:
    sys.path.insert(0, str(F2_ROOT))
if not (F2_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。fire_analysis.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import financial_model as fm
import fire_analysis as fa
import retirement_model as rm
import optimization as op
import fire_charts as fc

fc.setup_style()                     # 文字を大きくするなら fc.setup_style(1.3)
scenarios, settings = fm.load_scenarios()
named = {s["name"]: s for s in scenarios}
END_ACC = settings["accumulation_end_age"]      # 資産形成の計算終了年齢（80）
END_RET = settings["retirement_end_age"]        # リタイア後の計算終了年齢（100）
W = settings["withdrawal_rate"]                 # 取り崩し率（4%、教育用の仮定）

SAVE_FIGURES = False                 # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", fc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: (("not reached" if k == "fire_age" else "-") if r.get(k) is None else f.format(r[k]))
             for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F2_ROOT:", F2_ROOT)
print("シナリオ:", [s["label"] for s in scenarios], f"/ 取り崩し率 {W:.0%} / 資産形成 {END_ACC} 歳まで / リタイア後 {END_RET} 歳まで")

## モデル

$$ \text{Assets}_{t+1} = \text{Assets}_t \times (1 + r) - \text{Annual Expenses} $$

- 年初の資産に収益率を適用し、年末に生活費を取り崩す
- 足りない年は**あるだけ取り崩して 0 で打ち切る**（資産は負にならない）。足りなかった額は shortfall
- **資産枯渇年齢**：年末資産が 0 以下になった最初の年

## 1. FIRE 達成年齢でリタイアした場合

In [ ]:
#@title グラフを表示（コードは隠しています）
results = [op.evaluate_scenario(s, settings) for s in scenarios]
table(results, [("label", "scenario", "{}"), ("fire_age", "retire at (FIRE age)", "{}"),
                ("retire_assets", "assets", "{:,.0f}"), ("annual_expenses", "expenses", "{:,.0f}"),
                ("annual_return", "return", "{:.0%}"), ("depletion_age", "depletion age", "{}"),
                ("years_sustained", "years covered", "{}")])
series = []
for s, r in zip(scenarios, results):
    if r["fire_age"] is None:
        continue
    rows = rm.simulate_withdrawal(r["fire_age"], END_RET, r["retire_assets"], s["annual_expenses"], s["annual_return"])
    series.append((s["label"], rows, fc.color_of(s["name"])))
show(fc.plot_withdrawal(series, "Assets after retiring at the FIRE age", END_RET), "03_after_fire")

## RQ5：「目標に到達した」と「生活を続けられる」は違う

FIRE 目標 = 生活費 / 4% に到達しても、**リタイア後の収益率が 4% より低い**と、毎年の取り崩しが収益を上回り、資産は少しずつ減ります。
どの年齢でリタイアしたか（何年取り崩すか）によって、資産が持つかどうかが変わります。

In [ ]:
#@title グラフを表示（コードは隠しています）
assets0, expenses0 = 7500, 300                   # 目標ちょうど（300 / 4%）でリタイア
series = []
for ret, color in zip([0.02, 0.03, 0.04, 0.05], fc.ramp(4)):
    rows = rm.simulate_withdrawal(45, END_RET, assets0, expenses0, ret)
    dep = rm.depletion_age(rows)
    series.append((f"return {ret:.0%}" + ("" if dep is None else f" (out at {dep})"), rows, color))
show(fc.plot_withdrawal(series, "Retire at 45 with exactly 25x expenses", END_RET), "03_rq5_returns")

## 2. 取り崩し額（生活費）別の資産推移

In [ ]:
# TODO：リタイア開始年齢・資産・収益率を変えてみよう
retire_age, retire_assets, ret = 50, 7000, 0.03
series = []
for e, color in zip([200, 250, 300, 350, 400], fc.ramp(5)):
    series.append((f"expenses {e}", rm.simulate_withdrawal(retire_age, END_RET, retire_assets, e, ret), color))
show(fc.plot_withdrawal(series, f"Retire at {retire_age} with {retire_assets:,}: by annual expenses", END_RET),
     "03_by_expenses")
for label, rows, _c in series:
    print(f"{label}: 枯渇 {rm.depletion_age(rows)}  全額まかなえた年数 {rm.years_sustained(rows)}")

## 3. 収益率別の資産推移

In [ ]:
#@title グラフを表示（コードは隠しています）
series = []
for r, color in zip([0.0, 0.02, 0.04, 0.06], fc.ramp(4)):
    series.append((f"return {r:.0%}", rm.simulate_withdrawal(retire_age, END_RET, retire_assets, 300, r), color))
show(fc.plot_withdrawal(series, f"Retire at {retire_age} with {retire_assets:,}, expenses 300: by return", END_RET),
     "03_by_return")

## 4. リタイア開始年齢を変える

High Savings のシナリオで、資産形成を続けた後、いろいろな年齢でリタイアした場合の持続年数を比べます。

In [ ]:
s = named["high_savings"]
acc = fm.accumulate(s, END_ACC)
rows_out = []
for age in range(40, 66, 5):
    ret_rows = rm.simulate_withdrawal(age, END_RET, fm.assets_at(acc, age), s["annual_expenses"], s["annual_return"])
    rows_out.append({"age": age, "assets": fm.assets_at(acc, age), "depletion": rm.depletion_age(ret_rows),
                     "years": rm.years_sustained(ret_rows), "sustainable": rm.is_sustainable(ret_rows)})
table(rows_out, [("age", "retire at", "{}"), ("assets", "assets", "{:,.0f}"), ("depletion", "depletion age", "{}"),
                 ("years", "years covered", "{}"), ("sustainable", f"lasts to {END_RET}", "{}")])

## 学生向け TODO・考察

- [ ] リタイア開始年齢を変更する
- [ ] 生活費を変更する（年齢とともに変わる生活費にする）
- [ ] 収益率を変更する（リタイア後は低めにする、など）
- [ ] 資産持続期間を比較し、「FIRE 達成」の定義（RQ5）をチームで決める

**考察（チームで記入）**：TODO（目標到達と持続性の違い、4% ルールの限界、年金・インフレを入れるとどうなるか）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索・二分探索、第6章 挿入ソート）
- F2 リポジトリの `README.md`（モデルの仮定、4% ルールとその限界、計算量、モデルの限界と倫理的配慮）
- データ：`data/sample_scenarios.json`（この教材のための**架空のデータ**）